# Unsupervised Learning – Cycling Patterns in Zurich with k-Means, Hierarchical Clustering and PCA

**Dataset:** Bike counts in Zurich 2024–2025: daily number of bikes at 16 automatic counting stations of the city of
Zurich, with weather and calendar information (Open Data Stadt Zürich). Files: `../00_Data/zurich_bike_counts_2024_2025.csv`
and `../00_Data/zurich_bike_stations.csv` (names and directions of the stations).

**Questions:** Which counting stations have a similar traffic pattern – commuter routes, inner-city routes, leisure
routes? And what distinguishes one day from another? There is **no target variable** – we look for groups and for a
compact description of the 16 stations.

**Slides:** Part 05 «Unsupervised learning» – k-means, elbow method, hierarchical clustering (dendrogram), PCA.

**After this exercise you can ...**
- perform k-means iterations by hand and compute the within-cluster sum of squares,
- construct meaningful features for clustering,
- apply k-means (incl. scaling, elbow plot, silhouette analysis, several random starts) and interpret clusters,
- create and cut a dendrogram,
- use PCA to summarise many correlated features.

## Concept

The concepts behind this exercise (formulas, worked examples of the slides and additional explanations) are
explained in the notebook [clustering_pca_concept.ipynb](clustering_pca_concept.ipynb). Read it before you start.

## Libraries and settings

In [ ]:
# Libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples, adjusted_rand_score
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Show current working directory
print(os.getcwd())

## Exercise 1: k-means by hand

Six points in two dimensions: A (1, 1), B (2, 1), C (4, 3), D (5, 4), E (1, 2), F (5, 5).
We use $k = 2$ with the initial centres $\mu_1 = A = (1, 1)$ and $\mu_2 = C = (4, 3)$.

a) Assign every point to its nearest centre (Euclidean distance).

b) Compute the new centres.

c) Repeat the assignment step. Has the algorithm converged?

d) Compute the WCSS of the final solution.

e) Verify your results with `KMeans(n_clusters=2, init=np.array([[1, 1], [4, 3]]), n_init=1)`.

In [ ]:
points = pd.DataFrame({'x1': [1, 2, 4, 5, 1, 5], 'x2': [1, 1, 3, 4, 2, 5]}, index=list('ABCDEF'))
centres = np.array([[1.0, 1.0], [4.0, 3.0]])

for it in range(1, 4):
    dist = np.sqrt(((points.values[:, None, :] - centres[None, :, :])**2).sum(axis=2))
    labels = dist.argmin(axis=1)
    print(f'Iteration {it}: distances to centres =')
    print(pd.DataFrame(dist.round(2), index=points.index, columns=['to mu1', 'to mu2']).T)
    print('  assignment:', {p: int(l) + 1 for p, l in zip(points.index, labels)})
    new_centres = np.array([points.values[labels == c].mean(axis=0) for c in range(2)])
    print('  new centres:', new_centres.round(3).tolist(), '\n')
    if np.allclose(new_centres, centres):
        print('Converged: the centres do not change any more.')
        break
    centres = new_centres

wcss = sum(((points.values[labels == c] - centres[c])**2).sum() for c in range(2))
print('WCSS =', round(wcss, 3))

# Check with scikit-learn
km_check = KMeans(n_clusters=2, init=np.array([[1, 1], [4, 3]]), n_init=1).fit(points)
print('scikit-learn: centres =', km_check.cluster_centers_.round(3).tolist(), ' inertia =', round(km_check.inertia_, 3))

**Solution:**

a) Distances to $\mu_1=(1,1)$ / $\mu_2=(4,3)$: A 0 / 3.61, B 1 / 2.83, C 3.61 / 0, D 5 / 1.41, E 1 / 3.16, F 5.66 / 2.24
→ cluster 1 = {A, B, E}, cluster 2 = {C, D, F}.

b) $\mu_1 = \big(\tfrac{1+2+1}{3}, \tfrac{1+1+2}{3}\big) = (1.33, 1.33)$,
$\mu_2 = \big(\tfrac{4+5+5}{3}, \tfrac{3+4+5}{3}\big) = (4.67, 4)$.

c) With the new centres all points keep their cluster → the centres do not change any more; the algorithm has converged
after the second assignment.

d) Cluster 1: A $(0.11+0.11)$ + B $(0.44+0.11)$ + E $(0.11+0.44)$ = 1.33;
cluster 2: C $(0.44+1)$ + D $(0.11+0)$ + F $(0.11+1)$ = 2.67 → **WCSS = 4.0**.

## Exercise 2: Describe each counting station by its traffic pattern

To cluster the 16 counting stations, we describe each station by a few **ratios** that characterise *who* cycles there
– independently of how busy the station is:

| feature | definition | high value means … |
|---|---|---|
| `weekend_ratio` | mean on weekends / mean on working days | leisure traffic |
| `summer_winter_ratio` | mean on working days in May–September / in November–February | fair-weather / leisure traffic |
| `rain_ratio` | mean on rainy working days (> 4 h rain) / on dry working days | insensitive to rain (commuting) |
| `school_holiday_ratio` | mean on working days during / outside school holidays | not dependent on school and university |
| `temp_corr` | correlation of the daily counts with the temperature | weather-dependent traffic |

a) Import the bike counts and the station list (`../00_Data/zurich_bike_stations.csv`). Compute the five features for each
station (the code for the first two is given).

b) Show the table sorted by `weekend_ratio` and describe the extremes.

In [ ]:
# a) Import the data
bikes = pd.read_csv('../00_Data/zurich_bike_counts_2024_2025.csv', sep=',')
stations = pd.read_csv('../00_Data/zurich_bike_stations.csv', sep=',')
S = stations['column'].tolist()                      # the 16 station columns

work = (bikes['weekend'] == 0) & (bikes['holiday'] == 0)
summer, winter = bikes['month'].isin([5, 6, 7, 8, 9]), bikes['month'].isin([11, 12, 1, 2])
rain, dry = bikes['rain_duration_min'] > 240, bikes['rain_duration_min'] == 0

st = pd.DataFrame(index=S)
st['bikes_per_day'] = bikes[S].mean()
st['weekend_ratio'] = bikes.loc[bikes['weekend'] == 1, S].mean() / bikes.loc[work, S].mean()
st['summer_winter_ratio'] = bikes.loc[work & summer, S].mean() / bikes.loc[work & winter, S].mean()
st['rain_ratio'] = bikes.loc[work & rain, S].mean() / bikes.loc[work & dry, S].mean()
st['school_holiday_ratio'] = (bikes.loc[work & (bikes['school_holiday'] == 1), S].mean()
                              / bikes.loc[work & (bikes['school_holiday'] == 0), S].mean())
st['temp_corr'] = bikes[S].corrwith(bikes['temp_mean'])
st['name'] = stations.set_index('column')['name']

# b) Table sorted by weekend_ratio
print(st.sort_values('weekend_ratio').round(2).to_string())

**Solution:**

b) At the **Tödistrasse** only 42 % of the working-day traffic is counted on weekends; Hofwiesenstrasse, Mühlebachstrasse
and Bucheggplatz (≈ 50 %) are also typical commuter routes. At the **Mythenquai** the weekend traffic reaches 86 % of
the working-day traffic; it also has by far the highest summer/winter ratio (2.4), the strongest temperature dependence
(r ≈ 0.79) and the lowest rain ratio (0.50: on rainy days only half of the cyclists) – a leisure route along the lake.
The Lux-Guyer-Weg along the Limmat shows a similar but weaker pattern. Note that the features are measured on different
scales (ratios around 0.4–2.4, correlations 0.3–0.8) → standardise before clustering.

## Exercise 3: k-means – choosing k

a) Standardise the five ratio features (not `bikes_per_day` – we want to group stations by their *pattern*, not by
their size).

b) Fit `KMeans(n_clusters=k, n_init=10, random_state=42)` for $k = 1, \dots, 8$ and plot the WCSS (`inertia_`) against
$k$ (elbow plot). Additionally compute the silhouette score for $k = 2, \dots, 8$.

c) Which $k$ do you choose?

In [ ]:
features = ['weekend_ratio', 'summer_winter_ratio', 'rain_ratio', 'school_holiday_ratio', 'temp_corr']

# a) Standardise
X = StandardScaler().fit_transform(st[features])

# b) Elbow plot and silhouette score
ks = range(1, 9)
wcss, sil = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    wcss.append(km.inertia_)
    sil.append(silhouette_score(X, km.labels_) if k > 1 else np.nan)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ks, wcss, marker='o')
axes[0].set_xlabel('Number of clusters k')
axes[0].set_ylabel('WCSS (inertia)')
axes[0].set_title('Elbow plot')
axes[1].plot(ks, sil, marker='o', color='darkred')
axes[1].set_xlabel('Number of clusters k')
axes[1].set_ylabel('Silhouette score')
axes[1].set_title('Silhouette score')
for ax in axes:
    ax.grid(alpha=0.3)
plt.show()
print(pd.DataFrame({'k': ks, 'WCSS': np.round(wcss, 1), 'silhouette': np.round(sil, 3)}).to_string(index=False))

**Solution:**

The WCSS drops strongly from $k=1$ (80) to $k=2$ (39) and $k=3$ (22) and then decreases more slowly. The silhouette
score is highest for $k = 2$ (0.56: the two leisure routes vs. all others) and still clearly positive for $k = 3$ (0.42).
**$k = 3$** is a sensible choice: it is at the bend of the elbow and separates commuter routes from inner-city routes,
which is more informative for the city than the split into two groups. With only 16 stations, the choice remains a
domain decision.

## Exercise 4: Interpret the clusters

a) Fit k-means with $k = 3$ (`n_init=10, random_state=42`) and add the cluster labels to the station table.

b) Compute the mean of each feature per cluster and list the stations of each cluster. Give each cluster a meaningful
name.

c) Plot the average **weekly profile** (mean bikes per weekday, divided by the station's overall mean) of the three
clusters.

In [ ]:
# a) k-means with k = 3
km3 = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X)
st['cluster'] = km3.labels_

# b) Cluster profiles and stations
print(st.groupby('cluster')[features + ['bikes_per_day']].mean().round(2).to_string())
for c in range(3):
    print(f'Cluster {c}:', st.loc[st['cluster'] == c, 'name'].tolist())

# c) Weekly profile per cluster
weekdays = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
weekly = bikes.groupby('weekday')[S].mean().loc[weekdays] / bikes[S].mean()
plt.figure(figsize=(8, 4))
for c in range(3):
    plt.plot(weekdays, weekly[st.index[st['cluster'] == c]].mean(axis=1), marker='o', label=f'cluster {c}')
plt.axhline(1, color='grey', linewidth=1)
plt.ylabel('Bikes / station mean')
plt.title('Average weekly profile per cluster')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Solution:**

| cluster | typical values | name | stations |
|---|---|---|---|
| ≈ 6 stations | weekend ratio ≈ 0.49, strongest drop during school holidays (0.81), lowest temperature correlation | **commuter routes** | Mühlebachstrasse, Bucheggplatz, Hofwiesenstrasse, Militärbrücke, Tannenrauchstrasse, Tödistrasse |
| ≈ 8 stations | weekend ratio ≈ 0.65, highest volume (≈ 2'100 bikes per day) | **inner-city axes** (mixed traffic) | Langstrasse (4 stations), Hardbrücke (2), Limmatquai, Bertastrasse |
| 2 stations | weekend ratio ≈ 0.76, summer/winter ≈ 2.1, rain ratio ≈ 0.54, temperature correlation ≈ 0.76 | **leisure routes along the water** | Mythenquai, Lux-Guyer-Weg |

(The cluster numbers 0/1/2 are arbitrary.) The weekly profiles confirm the interpretation: the commuter routes have a
peak on Tuesday and drop to about half of their mean on Sundays, whereas the leisure routes lose much less traffic on
weekends (Saturday ≈ 0.86, Sunday ≈ 0.78 of their mean). On all routes Monday and Friday are weaker than Tuesday–Thursday – a sign of working
from home on these days.

## Exercise 5: Silhouette analysis

The mean silhouette score of Exercise 3 summarises a clustering in one number. A **silhouette analysis** looks at the
silhouette value $s_i = \frac{b_i - a_i}{\max(a_i, b_i)}$ of **every single station** ($a_i$ = mean distance to the
stations of its own cluster, $b_i$ = mean distance to the stations of the nearest other cluster):
$s_i$ close to 1 → well placed; close to 0 → between two clusters; negative → probably in the wrong cluster.

a) Compute the silhouette values of all stations with `silhouette_samples(X, labels)` for $k = 2, 3, 4$ and draw a
silhouette plot for each $k$: one horizontal bar per station, sorted within each cluster, with the station names and a
vertical line at the mean silhouette score (the plotting code is given).

b) Which stations are badly placed for $k = 3$? Compare the three plots. Does the silhouette analysis support $k = 3$?

In [ ]:
def silhouette_plot(ax, X, labels, names):
    """Horizontal silhouette plot: one bar per observation, sorted within clusters."""
    s = silhouette_samples(X, labels)
    y, ticks, ticklabels = 0, [], []
    for c in np.unique(labels):
        idx = np.where(labels == c)[0]
        idx = idx[np.argsort(s[idx])]
        ax.barh(range(y, y + len(idx)), s[idx], height=0.8, color=plt.cm.tab10(c), label=f'cluster {c}')
        ticks += list(range(y, y + len(idx)))
        ticklabels += [names[i] for i in idx]
        y += len(idx) + 1
    ax.axvline(s.mean(), color='black', linestyle='--', label=f'mean = {s.mean():.2f}')
    ax.set_yticks(ticks)
    ax.set_yticklabels(ticklabels, fontsize=8)
    ax.set_xlabel('silhouette value')
    ax.legend(fontsize=7, loc='best', framealpha=0.9)
    return s

# a) Silhouette plots for k = 2, 3, 4
names = st['name'].tolist()
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, k in zip(axes, [2, 3, 4]):
    labels_k = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X)
    s = silhouette_plot(ax, X, labels_k, names)
    ax.set_title(f'k = {k}')
plt.tight_layout()
plt.show()

# b) Silhouette values for k = 3
st['silhouette'] = silhouette_samples(X, km3.labels_)
print(st[['name', 'cluster', 'silhouette']].sort_values(['cluster', 'silhouette']).round(2).to_string(index=False))
print('Mean silhouette per cluster:')
print(st.groupby('cluster')['silhouette'].mean().round(2))

**Solution:**

a) and b) For $k = 3$ the **inner-city axes** form a compact cluster (all silhouette values ≈ 0.6). The **commuter routes**
are less homogeneous (0.14–0.42; Militärbrücke, Hofwiesenstrasse and Tödistrasse are close to the border). The striking
case is the **Lux-Guyer-Weg**: its silhouette value is **negative** (≈ −0.19) – on average it is closer to the
inner-city axes than to the Mythenquai, with which it shares its cluster. The leisure cluster is held together mainly by
the extreme Mythenquai (≈ 0.40).

- $k = 2$ (mean ≈ 0.56): 14 stations vs. the two leisure routes; the large cluster is compact, but the Lux-Guyer-Weg is
  again badly placed.
- $k = 4$ (mean ≈ 0.40): Mythenquai and Lux-Guyer-Weg become clusters of their own. A cluster with a single observation
  has by definition a silhouette value of 0 – such "clusters" are not meaningful.

Conclusion: the silhouette analysis shows that the three-cluster solution is solid for the commuter and the inner-city
group, but that the Lux-Guyer-Weg is a borderline case between "inner city" and "leisure". The mean silhouette score
alone (Exercise 3) would not have shown this. With only 16 stations, $k = 3$ remains a reasonable compromise; a
silhouette analysis is a valuable complement to the elbow plot – it shows *which* observations are badly clustered.

## Exercise 6: Stability of k-means

k-means depends on the random initial centres.

a) Fit k-means with $k = 3$ and `n_init=1` for `random_state` = 0, 1, …, 19. Print the WCSS of each run and compare the
cluster assignment with the solution of Exercise 4 using the *adjusted Rand index* (ARI: 1 = identical partition,
independent of the cluster numbers; ≈ 0 = random agreement).

b) What do you conclude? Why is `n_init=10` (as in the slides) a good idea?

In [ ]:
rows = []
for seed in range(20):
    km = KMeans(n_clusters=3, n_init=1, random_state=seed).fit(X)
    rows.append({'random_state': seed, 'WCSS': round(km.inertia_, 2),
                 'ARI vs. reference': round(adjusted_rand_score(km3.labels_, km.labels_), 3)})
stability = pd.DataFrame(rows)
print(stability.to_string(index=False))
print('\nReference solution (n_init=10): WCSS =', round(km3.inertia_, 2))

**Solution:**

Only one of the 20 runs with `n_init=1` finds the reference solution (WCSS ≈ 21.6, ARI = 1). Most runs end in a slightly
worse local minimum (WCSS ≈ 22.4, ARI ≈ 0.84 – one station switches the cluster), and four runs end in clearly worse
solutions (WCSS ≈ 31–33, ARI ≈ 0.1–0.3). k-means only guarantees a **local** optimum. With `n_init=10` the algorithm is
started 10 times and the solution with the smallest WCSS is kept – this makes the result much more reliable.

## Exercise 7: Hierarchical clustering and dendrogram

a) Compute the hierarchical clustering of the standardised features with `linkage(X, method='ward')` and draw the
dendrogram with the station names as labels.

b) Cut the dendrogram into 3 clusters (`fcluster(Z, t=3, criterion='maxclust')`) and compare them with the k-means
clusters (`pd.crosstab`).

c) At which height would you cut the dendrogram? Which stations are merged last?

In [ ]:
# a) Ward linkage and dendrogram
Z = linkage(X, method='ward')
plt.figure(figsize=(12, 5))
dendrogram(Z, labels=st['name'].tolist(), leaf_rotation=90, color_threshold=Z[-3, 2])
plt.ylabel('Merge distance (Ward)')
plt.title('Dendrogram of the 16 counting stations')
plt.show()

# b) Cut into 3 clusters and compare with k-means
st['hc_cluster'] = fcluster(Z, t=3, criterion='maxclust')
print(pd.crosstab(st['cluster'], st['hc_cluster'], rownames=['k-means'], colnames=['hierarchical']))
print('ARI k-means vs. hierarchical:', round(adjusted_rand_score(st['cluster'], st['hc_cluster']), 3))

**Solution:**

b) The hierarchical clustering (Ward) finds **exactly the same three groups** as k-means (ARI = 1.0) – a good sign that
the structure is real and not an artefact of one method.

c) The two leisure routes (Mythenquai, Lux-Guyer-Weg) are merged with the other stations only at the very end, at a
large distance – they are the most distinct group. A cut between the third-last and the second-last merge (the colours
in the plot) gives the 3 clusters; a cut higher up gives 2 clusters (leisure vs. all others), which matches the highest
silhouette score.

## Exercise 8: PCA – what makes days different?

Now we turn the perspective around: the **674 days** are the observations, the 16 stations are the features.

a) Standardise the daily counts of the 16 stations and fit a PCA. Plot the explained variance ratio of the first
components. How much variance does PC1 explain?

b) Show the loadings of PC1 and PC2 (`pca.components_`). Interpret both components.

c) Plot the days in the PC1–PC2 plane, coloured by weekend / working day. Which days have the highest and the lowest
PC2 scores?

In [ ]:
# a) PCA on the days
Z_days = StandardScaler().fit_transform(bikes[S])
pca = PCA()
scores = pca.fit_transform(Z_days)
evr = pca.explained_variance_ratio_
print('Explained variance ratio (PC1-PC5):', evr[:5].round(3), ' cumulative:', evr[:5].cumsum().round(3))

plt.figure(figsize=(7, 3.5))
pcs = [f'PC{i}' for i in range(1, 9)]
plt.bar(pcs, evr[:8], color='steelblue')
plt.ylabel('Share of explained variance')
plt.show()

# b) Loadings of PC1 and PC2
loadings = pd.DataFrame(pca.components_[:2].T, index=st['name'], columns=['PC1', 'PC2'])
print(loadings.round(2).sort_values('PC2').to_string())

# c) Days in the PC1-PC2 plane
plt.figure(figsize=(8, 5.5))
for w, label, color in [(0, 'working day', 'tab:blue'), (1, 'weekend', 'tab:red')]:
    m = bikes['weekend'] == w
    plt.scatter(scores[m, 0], scores[m, 1], s=12, alpha=0.6, color=color, label=label)
plt.xlabel(f'PC1 ({evr[0]:.0%} of variance)')
plt.ylabel(f'PC2 ({evr[1]:.0%} of variance)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

days = bikes[['date', 'weekday', 'temp_max', 'rain_duration_min', 'bikes_total']].assign(PC1=scores[:, 0], PC2=scores[:, 1])
print('Highest PC2:'); print(days.nlargest(5, 'PC2').round(1).to_string(index=False))
print('Lowest PC2:');  print(days.nsmallest(5, 'PC2').round(1).to_string(index=False))

**Solution:**

a) PC1 explains **89 %** of the variance, PC2 another 4 %. The 16 stations are so strongly correlated that one number
summarises most of the information about a day.

b) **PC1 = overall cycling volume:** all stations have almost the same positive loading (≈ 0.25). A day with a high PC1
score is a day with many cyclists everywhere (warm, dry working day). **PC2 = leisure vs. commuting:** the Mythenquai
(+0.74) and the Lux-Guyer-Weg (+0.40) load positively, commuter stations such as Tannenrauchstrasse and Militärbrücke
negatively. PC2 measures whether a day has *relatively* more leisure traffic than commuter traffic – exactly the
distinction found by the clustering.

c) Weekends have higher PC2 scores than working days. The highest PC2 scores are sunny summer **Sundays**
(e.g. 22 June 2025 with 33 °C), the lowest are rainy autumn **working days** (e.g. 23 October 2024 with more than 9 hours
of rain), when only commuters ride along their usual routes. Like on the slide «PCA», a few principal components are
enough to describe many correlated features.

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')